**Part 3 · LangGraph** · step 19 of 23  
[← 18 Tool-calling agent graph](18_tool_calling_agent_graph.ipynb) · [🏠 Roadmap](../README.md) · [20 When to use what →](../04_When_To_Use_What/20_choosing_the_right_tool.md)

# 19 · Research Chatbot with Multiple Real Tools

Same ReAct graph as notebook 18, but with real-world tools:
| Tool | Source | Needs |
|---|---|---|
| `search_arxiv` | custom `@tool` around the `arxiv` package | – |
| `wikipedia` | `WikipediaQueryRun` (langchain-community) | – |
| `tavily_search_results_json` | `TavilySearchResults` web search | `TAVILY_API_KEY` |

The LLM reads each tool's description and decides which one(s) to call.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

## Tools

In [ ]:
import arxiv
from langchain_core.tools import tool


@tool
def search_arxiv(query: str) -> str:
    """Search research papers on arXiv. Accepts keywords or an arXiv id such as 1706.03762."""
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=2)

    output = []
    for paper in client.results(search):
        authors = ", ".join(author.name for author in paper.authors)
        output.append(f"{paper.title}\nAuthors: {authors}\n\n{paper.summary}")
    return "\n\n".join(output)

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# top_k_results / doc_content_chars_max keep the tool output short (cheaper prompts)
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))
wiki.name

In [ ]:
# Deprecated in favour of `langchain-tavily` (TavilySearch) but still works.
# To upgrade: `uv add langchain-tavily` and `from langchain_tavily import TavilySearch`.
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

tools = [search_arxiv, wiki, tavily]

## LLM + tools: which tool does the model pick?

In [ ]:
from langchain_openai import ChatOpenAI

llm_with_tools = ChatOpenAI(model="gpt-4o").bind_tools(tools)

for question in ["What is the recent news in AI", "what is the latest research on quantum computing"]:
    response = llm_with_tools.invoke(question)
    print(question, "->", [(c["name"], c["args"]) for c in response.tool_calls])

## The graph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class State(TypedDict):
    """Chat history with the add_messages reducer."""

    messages: Annotated[list[AnyMessage], add_messages]


def tool_calling_llm(state: State) -> dict:
    """LLM node: answer directly or request one or more tool calls."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

In [ ]:
from IPython.display import Image, display


def show_graph(compiled_graph):
    """Render a compiled graph as a PNG.

    draw_mermaid_png() uses the mermaid.ink web service; if you are offline
    it falls back to printing the Mermaid source (paste it into mermaid.live).
    """
    try:
        display(Image(compiled_graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(compiled_graph.get_graph().draw_mermaid())


    show_graph(graph)

In [ ]:
messages = graph.invoke({"messages": [HumanMessage(content="1706.03762")]})
for m in messages["messages"]:
    m.pretty_print()

In [ ]:
messages = graph.invoke({"messages": [HumanMessage(content="What is the latest AI news, and tell me a research paper on quantum computing")]})
for m in messages["messages"]:
    m.pretty_print()

## Revision notes
- The ReAct loop is the same whatever the tools are — only the `tools` list changes.
- Good tool **descriptions** decide whether the model picks the right tool.
- `create_agent(model, tools)` builds exactly this graph for you; build it by hand when you need custom nodes/routing.

---
✅ **Done with 19 Research chatbot.** Next: [20 When to use what →](../04_When_To_Use_What/20_choosing_the_right_tool.md)  
[🏠 Roadmap](../README.md)